# TabICLv2 — CPU

Dùng **TabICLClassifier** và cố định checkpoint `tabicl-classifier-v2-20260212.ckpt`. Đây là TabICLv2, **không phải TabICL 2025 v1**. Tài liệu chính thức: https://github.com/soda-inria/tabicl .

**Lưu ý:** Chạy trên CPU, không pretrain từ đầu. Lần chạy đầu cần Internet để cài package/tải checkpoint. Nếu không có package/checkpoint, notebook ghi trạng thái lỗi; **không tự thay bằng mô hình giả lập**.

## 1. Cài thư viện
Chạy lệnh dưới đây trong Terminal hoặc bỏ dấu `#` ở cell tiếp theo. Nên dùng **Python 3.10–3.11** trong môi trường ảo riêng cho mỗi mô hình để tránh xung đột dependency.

In [3]:
%pip install 'tabicl>=1.0'

Defaulting to user installation because normal site-packages is not writeable
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tabicl]
Note: you may need to restart the kernel to use updated packages.


## 2. Đọc dữ liệu và chia tập thống nhất

In [1]:
from pathlib import Path
import os, json, time, traceback, platform, importlib.metadata
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
    if (ROOT.parent / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError('Mở notebook từ thư mục project hoặc chạy: os.chdir(duong_dan_giai_nen_ZIP)')
DATASET = ROOT / 'datasets' / 'breast_cancer_wisconsin.csv'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
SPLITS = ROOT / 'splits'; SPLITS.mkdir(exist_ok=True)
MODEL_ID = 'TabICLv2'
SEEDS = [42]                # Sau khi chạy thử: [42, 43, 44, 45, 46]
TRAIN_SIZES = [50, 100]    # Sau khi chạy thử: [50, 100, 200, 300]
N_CPU_THREADS = min(4, os.cpu_count() or 2)
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = str(N_CPU_THREADS)

frame = pd.read_csv(DATASET)
X = frame.drop(columns=['row_id', 'target']).to_numpy(dtype=np.float32)
y = frame['target'].to_numpy(dtype=np.int64)
assert len(np.unique(y)) == 2 and len(frame) == 569

def indices_for_seed(seed):
    path = SPLITS / f'breast_cancer_seed{seed}.json'
    if path.exists():
        s = json.loads(path.read_text(encoding='utf-8'))
    else:
        ids = np.arange(len(y))
        trainpool, test = train_test_split(ids, test_size=0.2, stratify=y, random_state=seed)
        train, val = train_test_split(trainpool, test_size=0.2, stratify=y[trainpool], random_state=seed)
        s = {'seed': seed, 'train': sorted(map(int,train)), 'val': sorted(map(int,val)), 'test': sorted(map(int,test))}
        path.write_text(json.dumps(s,indent=2),encoding='utf-8')
    assert not (set(s['train']) & set(s['test']))
    assert not (set(s['val']) & set(s['test']))
    assert not (set(s['train']) & set(s['val']))
    return s

def make_fold(seed, n):
    s = indices_for_seed(seed)
    pool = np.asarray(s['train'],dtype=int)
    if n > len(pool):
        raise ValueError(f'n_train={n} > train pool {len(pool)}')
    chosen, _ = train_test_split(pool, train_size=n, stratify=y[pool], random_state=seed + 101)
    chosen = np.sort(chosen)
    return (X[chosen], y[chosen], X[s['val']], y[s['val']], X[s['test']], y[s['test']], chosen, s)

def metrics(y_true, y_pred, prob):
    p = np.asarray(prob,dtype=float)
    assert p.ndim == 2 and p.shape == (len(y_true),2), p.shape
    p = np.clip(p, 1e-8, 1.0); p = p / p.sum(axis=1,keepdims=True)
    return {'accuracy':accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,average='macro',zero_division=0),
            'roc_auc':roc_auc_score(y_true,p[:,1]),
            'log_loss':log_loss(y_true,p,labels=[0,1])}

def record(status, seed, n, fit_sec=None, pred_sec=None, scores=None, err='', version=''):
    out = {'model':MODEL_ID,'dataset':'breast_cancer_wisconsin', 'seed':seed,
           'train_size':n,'status':status,'accuracy':np.nan,'macro_f1':np.nan,
           'roc_auc':np.nan,'log_loss':np.nan,'fit_seconds':fit_sec,
           'predict_seconds':pred_sec,'package_version':version,
           'error':str(err)[:800]}
    if scores: out.update(scores)
    return out

def save_results(rows):
    file=RESULTS / (MODEL_ID.lower().replace(' ','_')+'.csv')
    pd.DataFrame(rows).to_csv(file,index=False)
    print('Đã ghi:',file)
    display(pd.DataFrame(rows))
    return file

print('DATASET:', DATASET, '| shape:',X.shape,'| classes:',np.bincount(y))
print('CPU:',platform.processor() or platform.machine(), '| threads:',N_CPU_THREADS)
print('Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.')

DATASET: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/datasets/breast_cancer_wisconsin.csv | shape: (569, 30) | classes: [212 357]
CPU: arm | threads: 4
Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.


## 3. Chạy mô hình gốc trên CPU và xuất kết quả

In [5]:
import torch
from tabicl import TabICLClassifier
import importlib.metadata
assert not torch.cuda.is_available() or True  # luôn ép model chạy trên CPU
rows=[]
for seed in SEEDS:
    for n in TRAIN_SIZES:
        xtr,ytr,xval,yval,xte,yte,ids,split=make_fold(seed,n)
        try:
            model=TabICLClassifier(device='cpu', n_estimators=1, batch_size=1,
                use_amp=False, use_fa3=False, offload_mode='auto',
                checkpoint_version='tabicl-classifier-v2-20260212.ckpt',random_state=seed)
            t=time.perf_counter(); model.fit(xtr,ytr); fit_sec=time.perf_counter()-t
            t=time.perf_counter(); prob=model.predict_proba(xte); pred_sec=time.perf_counter()-t
            pred=model.classes_[np.argmax(prob,axis=1)] if hasattr(model,'classes_') else np.argmax(prob,axis=1)
            rows.append(record('ok',seed,n,fit_sec,pred_sec,metrics(yte,pred,prob),version=importlib.metadata.version('tabicl')))
        except Exception as e:
            rows.append(record('failed',seed,n,err=repr(e),version=importlib.metadata.version('tabicl')))
            print(f'FAILED seed={seed}, n={n}:',repr(e))
file=save_results(rows)

'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: a75ac397-7e39-4b9b-9875-27013ae525f1)')' thrown while requesting HEAD https://huggingface.co/jingang/TabICL/resolve/main/tabicl-classifier-v2-20260212.ckpt
Retrying in 1s [Retry 1/5].


INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 57314ccc-3318-43b5-89ac-187264369cc3)')' thrown while requesting HEAD https://huggingface.co/jingang/TabICL/resolve/main/tabicl-classifier-v2-20260212.ckpt
Retrying in 2s [Retry 2/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 8e49f9fd-59e0-4135-8968-768fa559ff95)')' thrown while requesting HEAD https://huggingface.co/jingang/TabICL/resolve/main/tabicl-classifier-v2-20260212.ckpt
Retrying in 4s [Retry 3/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 1df9a7d6-525a-41e8-8c18-e6dac720224a)')' thrown while requesting HEAD https://huggingface.co/jingang/TabICL/resolve/main/tabicl-classifier-v2-20260212.ckpt
Retrying in 8s [Retry 4/5].


Đã ghi: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/results/tabiclv2.csv


,model,dataset,seed,train_size,status,accuracy,macro_f1,roc_auc,log_loss,fit_seconds,predict_seconds,package_version,error
0,TabICLv2,breast_cancer_wisconsin,42,50,ok,0.973684,0.971863,0.996032,0.083052,25.869829,0.062738,2.2.0,
1,TabICLv2,breast_cancer_wisconsin,42,100,ok,0.973684,0.971277,0.997685,0.064979,0.070166,0.067117,2.2.0,


## 4. Ghi chú và giới hạn

`n_estimators=1` giảm thời gian CPU nhưng không phải cấu hình mặc định của paper. Muốn so sánh với kết quả paper, phải chạy lại ensemble và protocol tương ứng. Không coi một seed trên một dataset là bằng chứng SOTA.